<a href="https://colab.research.google.com/github/gitBarrettJones/CIS115-Spring2026-Assignments/blob/main/Week15AI2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gardening Inventory Tool

In [6]:
import sqlite3
from datetime import datetime, timedelta

DATABASE_NAME = 'gardening_inventory.db'

def create_database_and_table():
    """Creates the SQLite database and the plants table if they don't exist."""
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS plants (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL UNIQUE,
            type TEXT NOT NULL,
            watering_frequency_days INTEGER NOT NULL,
            last_watered_date TEXT NOT NULL,
            next_watering_date TEXT
        )
    ''')
    conn.commit()
    conn.close()
    print(f"Database '{DATABASE_NAME}' and 'plants' table ensured.")

# Initialize the database
create_database_and_table()

Database 'gardening_inventory.db' and 'plants' table ensured.


Now that the database is set up, let's create functions to add new plants and update their watering status.

In [15]:
def calculate_next_watering_date(last_watered_str, frequency_days):
    """Calculates the next watering date."""
    last_watered = datetime.strptime(last_watered_str, '%Y-%m-%d').date()
    next_watering = last_watered + timedelta(days=frequency_days)
    return next_watering.strftime('%Y-%m-%d')

def add_plant(name, plant_type, watering_frequency_days, last_watered_date):
    """Adds a new plant to the database."""
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        if not name or not plant_type:
            print("Error: Plant name and type cannot be empty.")
            return
        # Enhanced validation for name and type: check if they are purely numeric or contain numbers inappropriately
        if name.replace(' ', '').isdigit(): # Allow spaces, but check if remaining chars are all digits
            print("Error: Plant name cannot be purely numeric.")
            return
        if plant_type.replace(' ', '').isdigit(): # Allow spaces, but check if remaining chars are all digits
            print("Error: Plant type cannot be purely numeric.")
            return
        if not isinstance(watering_frequency_days, int) or watering_frequency_days <= 0:
            print("Error: Watering frequency must be a positive integer.")
            return

        next_watering = calculate_next_watering_date(last_watered_date, watering_frequency_days)
        cursor.execute(
            "INSERT INTO plants (name, type, watering_frequency_days, last_watered_date, next_watering_date) VALUES (?, ?, ?, ?, ?)",
            (name, plant_type, watering_frequency_days, last_watered_date, next_watering)
        )
        conn.commit()
        print(f"Plant '{name}' added successfully. Next watering due: {next_watering}")
    except sqlite3.IntegrityError:
        print(f"Error: Plant with name '{name}' already exists.")
    except ValueError as e:
        print(f"Error with date format: {e}. Please use YYYY-MM-DD.")
    except Exception as e:
        print(f"An unexpected error occurred while adding plant '{name}': {e}")
    finally:
        conn.close()

def update_last_watered(plant_name, new_last_watered_date):
    """Updates the last watered date and recalculates the next watering date for a plant."""
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("SELECT watering_frequency_days FROM plants WHERE name = ?", (plant_name,))
        result = cursor.fetchone()
        if result:
            frequency_days = result[0]
            next_watering = calculate_next_watering_date(new_last_watered_date, frequency_days)
            cursor.execute(
                "UPDATE plants SET last_watered_date = ?, next_watering_date = ? WHERE name = ?",
                (new_last_watered_date, next_watering, plant_name)
            )
            conn.commit()
            print(f"'{plant_name}' updated. New last watered: {new_last_watered_date}, Next watering due: {next_watering}")
        else:
            print(f"Plant '{plant_name}' not found.")
    except ValueError as e:
        print(f"Error with date format: {e}. Please use YYYY-MM-DD.")
    except Exception as e:
        print(f"An unexpected error occurred while updating plant '{plant_name}': {e}")
    finally:
        conn.close()

def remove_plant(plant_name):
    """Removes a plant from the database."""
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("DELETE FROM plants WHERE name = ?", (plant_name,))
        if cursor.rowcount > 0:
            conn.commit()
            print(f"Plant '{plant_name}' removed successfully.")
        else:
            print(f"Plant '{plant_name}' not found.")
    except Exception as e:
        print(f"An unexpected error occurred while removing plant '{plant_name}': {e}")
    finally:
        conn.close()

Let's add some example plants to your inventory and display them.

In [13]:
# Example: Add some plants (using dates from 2026 to make them more current)
# To avoid 'already exists' errors on re-run, you might first clear the table or ensure unique names.
# For demonstration, let's use slightly modified names or clear the table if running repeatedly.

# A simpler approach to avoid errors for demonstration is to use slightly different names
# or ensure the table is empty before adding.
# For this demonstration, I'll add a check to clear the table first for clean re-runs.

def clear_plants_table():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM plants")
    conn.commit()
    conn.close()
    print("Plants table cleared for fresh example data.")

clear_plants_table()

add_plant('Tomato Plant 2026', 'Vegetable', 3, '2026-04-13')
add_plant('Basil Herb 2026', 'Herb', 2, '2026-04-14')
add_plant('Rose Bush 2026', 'Flower', 7, '2026-04-09')
add_plant('Fern Houseplant 2026', 'Houseplant', 5, '2026-04-11')

# Example: Update a plant's last watered date
update_last_watered('Tomato Plant 2026', '2026-04-16')

Plants table cleared for fresh example data.
Plant 'Tomato Plant 2026' added successfully. Next watering due: 2026-04-16
Plant 'Basil Herb 2026' added successfully. Next watering due: 2026-04-16
Plant 'Rose Bush 2026' added successfully. Next watering due: 2026-04-16
Plant 'Fern Houseplant 2026' added successfully. Next watering due: 2026-04-16
'Tomato Plant 2026' updated. New last watered: 2026-04-16, Next watering due: 2026-04-19


In [9]:
import pandas as pd

def get_all_plants():
    """Retrieves all plants from the database and returns them as a Pandas DataFrame."""
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql_query("SELECT * FROM plants", conn)
    conn.close()
    return df

def display_plants():
    """Displays the current plant inventory."""
    plants_df = get_all_plants()
    if not plants_df.empty:
        print("\n--- Current Plant Inventory ---")
        display(plants_df)
    else:
        print("Your plant inventory is empty.")

display_plants()


--- Current Plant Inventory ---


,id,name,type,watering_frequency_days,last_watered_date,next_watering_date
0,1,Tomato Plant,Vegetable,3,2023-10-27,2023-10-30
1,2,Basil,Herb,2,2023-10-26,2023-10-28
2,3,Rose,Flower,7,2023-10-20,2023-10-27
3,4,Fern,Houseplant,5,2023-10-23,2023-10-28


Next, let's implement the logic to check for plants that are overdue for watering and provide an alert function.

In [10]:
def check_overdue_plants():
    """Checks for plants that are overdue for watering and returns them."""
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    today = datetime.now().date()
    overdue_plants = []

    cursor.execute("SELECT name, next_watering_date FROM plants")
    plants = cursor.fetchall()
    conn.close()

    for name, next_watering_str in plants:
        next_watering = datetime.strptime(next_watering_str, '%Y-%m-%d').date()
        if next_watering <= today:
            overdue_plants.append({'name': name, 'next_watering_due': next_watering_str})

    return pd.DataFrame(overdue_plants) if overdue_plants else pd.DataFrame(columns=['name', 'next_watering_due'])

def alert_overdue_plants():
    """Prints an alert for plants that are overdue for watering."""
    overdue_df = check_overdue_plants()
    if not overdue_df.empty:
        print("\n--- Watering Alerts! ---")
        print("The following plants are overdue for watering:")
        display(overdue_df)
    else:
        print("\nAll plants are watered up to date!")

alert_overdue_plants()


--- Watering Alerts! ---
The following plants are overdue for watering:


,name,next_watering_due
0,Tomato Plant,2023-10-30
1,Basil,2023-10-28
2,Rose,2023-10-27
3,Fern,2023-10-28


Let's add interactive input forms for easier data entry and updates.

In [20]:
import ipywidgets as widgets
from IPython.display import display

def create_add_plant_form():
    """Creates and displays an interactive form to add a new plant."""
    print("\n--- Add New Plant ---")
    name_input = widgets.Text(description='Plant Name:')
    type_input = widgets.Text(description='Plant Type:')
    frequency_input = widgets.IntSlider(description='Watering Freq (days):', min=1, max=30, value=7)
    last_watered_input = widgets.DatePicker(description='Last Watered Date:')
    submit_button = widgets.Button(description='Add Plant')

    output = widgets.Output()

    def on_submit_add(b):
        with output:
            output.clear_output()
            if not last_watered_input.value:
                print("Error: Last Watered Date cannot be empty.")
                return
            add_plant(
                name_input.value,
                type_input.value,
                frequency_input.value,
                last_watered_input.value.strftime('%Y-%m-%d')
            )
            # Re-display relevant forms to refresh dropdowns and plant list
            create_update_plant_form()
            create_remove_plant_form()
            create_display_plants_button()

    submit_button.on_click(on_submit_add)

    display(name_input, type_input, frequency_input, last_watered_input, submit_button, output)

def create_update_plant_form():
    """Creates and displays an interactive form to update a plant's last watered date."""
    print("\n--- Update Plant Watering ---")
    plants_df = get_all_plants()
    if plants_df.empty:
        print("No plants in inventory to update.")
        return

    plant_names = plants_df['name'].tolist()
    plant_selector = widgets.Dropdown(
        options=plant_names,
        description='Select Plant:'
    )
    new_last_watered_input = widgets.DatePicker(description='New Last Watered Date:')
    update_button = widgets.Button(description='Update Watering')

    output = widgets.Output()

    def on_submit_update(b):
        with output:
            output.clear_output()
            if not new_last_watered_input.value:
                print("Error: New Last Watered Date cannot be empty.")
                return
            update_last_watered(
                plant_selector.value,
                new_last_watered_input.value.strftime('%Y-%m-%d')
            )
            display_plants()
            alert_overdue_plants()

    update_button.on_click(on_submit_update)

    display(plant_selector, new_last_watered_input, update_button, output)

def create_display_plants_button():
    """Creates a button to display the current plant inventory and alerts."""
    print("\n--- View Current Inventory ---")
    display_button = widgets.Button(description='Show Plant List')
    output = widgets.Output()

    def on_display_click(b):
        with output:
            output.clear_output()
            display_plants()
            alert_overdue_plants()

    display_button.on_click(on_display_click)
    display(display_button, output)

def create_remove_plant_form():
    """Creates and displays an interactive form to remove a plant."""
    print("\n--- Remove Plant ---")
    plants_df = get_all_plants()
    if plants_df.empty:
        print("No plants in inventory to remove.")
        return

    plant_names = plants_df['name'].tolist()
    plant_selector = widgets.Dropdown(
        options=plant_names,
        description='Select Plant:'
    )
    remove_button = widgets.Button(description='Remove Plant')

    output = widgets.Output()

    def on_submit_remove(b):
        with output:
            output.clear_output()
            remove_plant(plant_selector.value)
            # Only re-display the plants button; do not re-render other forms to avoid clutter.
            # The dropdowns in existing update/remove forms will become stale.
            create_display_plants_button()

    remove_button.on_click(on_submit_remove)

    display(plant_selector, remove_button, output)

# Display the forms and button
create_add_plant_form()
create_update_plant_form()
create_remove_plant_form()
create_display_plants_button()


--- Add New Plant ---


Text(value='', description='Plant Name:')

Text(value='', description='Plant Type:')

IntSlider(value=7, description='Watering Freq (days):', max=30, min=1)

DatePicker(value=None, description='Last Watered Date:')

...

Button(description='Remove Plant', style=ButtonStyle())

Output()


--- View Current Inventory ---


Button(description='Show Plant List', style=ButtonStyle())

Output()

In [21]:
# Install nbformat to handle notebook files programmatically
!pip install nbformat

In [22]:
import nbformat

# IMPORTANT: Replace 'your_notebook.ipynb' with the actual name of your notebook file.
# You can find the current notebook name by looking at the tab title in your browser,
# or by listing files in the current directory (e.g., !ls).
notebook_name = "your_notebook.ipynb"

try:
    nb = nbformat.read(notebook_name, as_version=4)
    nb.metadata.pop("widgets", None)
    nbformat.write(nb, notebook_name)
    print(f"Successfully removed widget metadata from '{notebook_name}'.")
    print("You may need to refresh the Colab page or restart the runtime for changes to take effect.")
except FileNotFoundError:
    print(f"Error: Notebook '{notebook_name}' not found. Please ensure the name is correct and the notebook is saved.")
except Exception as e:
    print(f"An error occurred while processing the notebook: {e}")

Error: Notebook 'your_notebook.ipynb' not found. Please ensure the name is correct and the notebook is saved.
